# 🏆 NVIDIA Nemotron-3 Reasoning Challenge
### RTX PRO 6000 (Blackwell SM 12.0) — Upgraded SFT + GRPO Pipeline

**Required data sources (add via right panel → Add data):**
1. Competition dataset → `nvidia-nemotron-3-reasoning-challenge`
2. Offline packages → `nvidia-nemotron-offline-packages`
3. Model weights → Models → `metric/nemotron-3-nano-30b-a3b-bf16/transformers/default`
4. Utility script → Notebooks → `ryanholbrook/nvidia-utility-script` ← **CRITICAL for Blackwell ptxas**

**What this notebook does:**
- **Phase 0:** EDA — puzzle classification + answer distribution
- **Phase 1:** SFT — Supervised Fine-Tuning on gold answers with LoRA r=32
- **Phase 2:** GRPO — Group Relative Policy Optimisation with three reward signals
- **Phase 3:** Save, validate, and package `submission.zip`

All Triton / Blackwell bugs are fixed before any model load.


## 1. Offline Package Installation

> Only `datasets` and `trl` need updating. All other packages (`torch`, `transformers`, `peft`, `triton`, `bitsandbytes`, `kagglehub`) come from the Kaggle Docker image and must **not** be reinstalled — they carry Blackwell-specific patches.

In [ ]:
# ─── Install ONLY what is missing from the Kaggle Docker image ────────────────
# torch, transformers, peft, triton, bitsandbytes, kagglehub are PRE-INSTALLED
# in dockerImageVersionId 31287. DO NOT reinstall them — they contain
# Blackwell-specific patches from Kaggle/NVIDIA.
!pip install -q --no-index \
    --find-links /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages \
    datasets trl \
    --ignore-installed

import datasets, trl
print(f'datasets : {datasets.__version__}')   # expect 4.8.4
print(f'trl      : {trl.__version__}')         # expect 0.29.1


## 2. Imports & Global Configuration

In [ ]:
import os, sys, stat, shutil, gc, re, json, types, zipfile
from collections import Counter

import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig, GRPOTrainer, GRPOConfig

# Required for the Triton version spoof applied before each trainer.train()
import triton.backends.nvidia.compiler as nv_compiler

# ─── GPU diagnostics ──────────────────────────────────────────────────────────
print(f'torch        : {torch.__version__}')
print(f'GPU          : {torch.cuda.get_device_name(0)}')
print(f'VRAM total   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
COMPUTE_DTYPE = torch.bfloat16

# ─── Hyperparameters ──────────────────────────────────────────────────────────
LORA_RANK          = 32          # max allowed by competition
SFT_MAX_LEN        = 512         # token budget per sample
SFT_EPOCHS         = 2
SFT_LR             = 2e-4
SFT_GRAD_ACCUM     = 8           # effective batch = 8
GRPO_EPOCHS        = 2
GRPO_LR            = 5e-6
GRPO_NUM_GEN       = 8           # completions sampled per prompt
GRPO_MAX_COMP_LEN  = 512
OUTPUT_DIR         = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('\nHyperparameters loaded ✅')


## 3. Bug Fixes — MUST Run Before Any Model Loading

> Five fixes are required for Nemotron-H on a Blackwell GPU. Run this cell first, every session.

In [ ]:
import sys
# ─────────────────────────────────────────────────────────────────────────────
# FIX 1: Bypass Triton RMSNorm kernel
# Nemotron-H uses Mamba SSM blocks with a Triton-accelerated rmsnorm_fn.
# On Blackwell (SM 12.0) that kernel cannot compile yet.
# Replace with an identical pure-PyTorch fallback.
# ─────────────────────────────────────────────────────────────────────────────
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

modules_patched = 0
for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn
        modules_patched += 1
print(f'Fix 1 ✅  RMSNorm bypassed ({modules_patched} modules patched)')

# ─────────────────────────────────────────────────────────────────────────────
# FIX 2: Copy PTXAS binary to /tmp and make it executable
# The utility script is installed at /kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script
# The binary is read‑only there, so we copy it to /tmp and fix permissions.
# ─────────────────────────────────────────────────────────────────────────────
ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
ptxas_dst = '/tmp/ptxas-blackwell'

try:
    if os.path.exists(ptxas_src):
        # Remove any existing file to avoid permission conflicts
        if os.path.exists(ptxas_dst):
            os.remove(ptxas_dst)
        
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, 0o755)  # rwxr-xr-x

        import triton.backends.nvidia as nv_backend
        src_bin = os.path.join(os.path.dirname(nv_backend.__file__), 'bin')
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, 0o755)

        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst
        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst
        print('Fix 2 ✅  PTXAS binary copied to /tmp and env vars set')
    else:
        print(f'Fix 2 ❌  PTXAS binary not found at expected location: {ptxas_src}')
        print('    Please ensure "ryanholbrook/nvidia-utility-script" is added via "Add Data" (Notebooks tab).')
except Exception as e:
    print(f'Fix 2 ❌  {e}')
                

# ─────────────────────────────────────────────────────────────────────────────
# FIX 3: Stub out mamba_ssm.modules.mamba3
# Nemotron-H optionally imports Mamba3 which triggers a cutlass C++ extension
# not compiled for Blackwell SM 12.0. A dummy stub prevents the crash.
# ─────────────────────────────────────────────────────────────────────────────
try:
    mamba3_stub = types.ModuleType('mamba_ssm.modules.mamba3')
    class DummyMamba3:
        def __init__(self, *a, **kw):
            raise RuntimeError('Mamba3 stub — should not be instantiated')
    mamba3_stub.Mamba3 = DummyMamba3
    sys.modules['mamba_ssm.modules.mamba3'] = mamba3_stub
    print('Fix 3 ✅  mamba_ssm.modules.mamba3 stubbed')
except Exception as e:
    print(f'Fix 3 ❌  {e}')

print('\nAll pre-load fixes applied ✅')


In [ ]:
import os
input_dir = '/kaggle/input'
if os.path.exists(input_dir):
    print("Contents of /kaggle/input:")
    for item in os.listdir(input_dir):
        print(f"  {item}")
else:
    print("/kaggle/input not found")

## 4. EDA — Data Loading & Puzzle Analysis

In [ ]:
train_df = pd.read_csv('/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv')
test_df  = pd.read_csv('/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/test.csv')

print(f'Train: {len(train_df)} rows | Test: {len(test_df)} rows')
print(f'Columns: {list(train_df.columns)}')
display(train_df.head(3))


In [ ]:
def classify_puzzle(prompt: str) -> str:
    p = prompt.lower()
    if any(k in p for k in ['xor', 'and ', 'or ', 'shift', 'bit', 'binary']): return 'bit_manipulation'
    if any(k in p for k in ['equation', 'solve', 'variable', 'algebra']):      return 'algebra'
    if any(k in p for k in ['sequence', 'pattern', 'next']):                   return 'sequence'
    if any(k in p for k in ['prime', 'divisor', 'modulo', 'gcd', 'lcm']):      return 'number_theory'
    return 'other'

train_df['puzzle_type'] = train_df['prompt'].apply(classify_puzzle)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
counts = train_df['puzzle_type'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#5DCAA5','#7F77DD','#D85A30','#888780','#F5C518'])
axes[0].set_title('Puzzle type distribution')
axes[0].tick_params(axis='x', rotation=20)

def try_float(v):
    try: return float(v)
    except: return None

nums = train_df['answer'].apply(try_float).dropna()
axes[1].hist(nums.clip(-200, 500), bins=50, color='#378ADD', edgecolor='none')
axes[1].set_title('Answer distribution (clipped −200 to 500)')
plt.tight_layout()
plt.savefig('/kaggle/working/eda.png', dpi=110)
plt.show()
print(train_df['puzzle_type'].value_counts())


## 5. Tokenizer & Model Loading

> Full bfloat16, single GPU. With 96 GB VRAM no quantisation is needed.
> `device_map={'': 0}` forces all layers onto GPU 0 — no CPU offload, maximum throughput.

In [ ]:
MODEL_PATH = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
print(f'Model path: {MODEL_PATH}')

# ─── Tokenizer ────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = 'right'   # required for SFT packing=False

# ─── Model ────────────────────────────────────────────────────────────────────
print('Loading model (may take ~3 min)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map        = {'': 0},        # all layers on GPU 0
    trust_remote_code = True,
    torch_dtype       = torch.bfloat16,
    local_files_only  = True,
    use_safetensors   = True,
)
print(f'VRAM used after load: {torch.cuda.memory_allocated(0)/1e9:.1f} GB')

# ─── Fix 4: Disable Nemotron-H fast path (broken CUDA kernels on Blackwell) ──
# Must be applied AFTER model load
patched_mods = []
for name, mod in sys.modules.items():
    if 'modeling_nemotron_h' in name:
        mod.is_fast_path_available = False
        patched_mods.append(name)
print(f'Fix 4 ✅  Fast path disabled in: {patched_mods}')


## 6. LoRA Setup

> Nemotron-H is a **Mamba-H hybrid** — NOT a standard transformer. SSM blocks use `in_proj` / `out_proj` instead of `q_proj` / `k_proj` / `v_proj`. The target_modules must be a regex string.

In [ ]:
# ─── LoRA Configuration ───────────────────────────────────────────────────────
lora_config = LoraConfig(
    r              = LORA_RANK,
    lora_alpha     = 32,
    # Regex matches Mamba-H SSM layers + MLP projection layers
    target_modules = r'.*\.(in_proj|out_proj|up_proj|down_proj)$',
    lora_dropout   = 0.05,
    bias           = 'none',
    task_type      = TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expect: ~880M trainable / 32.4B total (~2.7%)
print('✅ LoRA adapter attached')


## 7. Data Formatting for SFT

In [ ]:
def extract_boxed(text: str):
    """Extract answer from \\boxed{...}, fall back to last number in text."""    
    m = re.search(r'\\boxed\{([^}]+)\}', text)
    if m: return m.group(1).strip()
    nums = re.findall(r'-?\d+(?:\.\d+)?', text)
    return nums[-1] if nums else None


SYSTEM_PROMPT = (
    'You are a precise logical-reasoning assistant. '
    'Think step by step. Place your final answer in \\boxed{}.'
)

def format_for_sft(example: dict) -> dict:
    """Format each training row into Nemotron chat template."""    
    user_msg      = example['prompt'] + '\nPut your final answer inside \\boxed{}.'
    assistant_msg = f"\\boxed{{{example['answer']}}}"

    messages = [
        {'role': 'user',      'content': user_msg},
        {'role': 'assistant', 'content': assistant_msg},
    ]
    try:
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    except Exception:
        text = (
            f'<|im_start|>user\n{user_msg}<|im_end|>\n'
            f'<|im_start|>assistant\n{assistant_msg}<|im_end|>'
        )
    return {'text': text}


hf_dataset = Dataset.from_pandas(train_df[['prompt', 'answer']])
hf_dataset = hf_dataset.map(format_for_sft, remove_columns=hf_dataset.column_names)
hf_dataset1 = hf_dataset.shuffle(seed=42).select(range(100))

print(f'SFT dataset: {len(hf_dataset)} samples')
print('\nSample (first 500 chars):')
print(hf_dataset1[0]['text'][:500])


## 8. Phase 1 — SFT Training

> Fix 5 (Triton env vars + version spoof) must be set **before** `trainer.train()` to prevent ptxas failures during the first JIT compilation.

In [ ]:
# ─── Fix 5: Triton env vars + version spoof ───────────────────────────────────
# Must be set before the first Triton JIT compilation (= first training step)
os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = '/tmp/ptxas-blackwell'
nv_compiler.get_ptxas_version = lambda arch: '12.0'
print('Fix 5 ✅  Triton env vars set')

# ─── SFT Configuration ────────────────────────────────────────────────────────
sft_config = SFTConfig(
    output_dir                    = OUTPUT_DIR,
    per_device_train_batch_size   = 1,
    gradient_accumulation_steps   = SFT_GRAD_ACCUM,
    gradient_checkpointing        = True,
    gradient_checkpointing_kwargs = {'use_reentrant': False},
    learning_rate                 = SFT_LR,
    num_train_epochs              = SFT_EPOCHS,
    logging_steps                 = 10,
    save_strategy                 = 'no',
    fp16                          = False,
    bf16                          = True,
    optim                         = 'adamw_torch',   # no BNB dependency
    lr_scheduler_type             = 'cosine',
    warmup_ratio                  = 0.1,
    max_grad_norm                 = 1.0,
    report_to                     = 'none',
    dataset_text_field            = 'text',
    max_length                    = SFT_MAX_LEN,
    packing                       = False,
)

# ─── SFTTrainer ───────────────────────────────────────────────────────────────
# processing_class=tokenizer  ←  correct TRL 0.29 API (tokenizer= is deprecated)
sft_trainer = SFTTrainer(
    model            = model,
    train_dataset    = hf_dataset1,
    args             = sft_config,
    processing_class = tokenizer,
)

print('🏋️  Starting SFT...')
sft_trainer.train()
print('✅ SFT complete')


In [ ]:
!find /kaggle -name "*.safetensors" -o -name "adapter_*" 2>/dev/null | head -20

In [ ]:
# Try saving from the trainer (most likely still exists)
sft_trainer.model.save_pretrained('/kaggle/working')
print("✅ Adapter saved from trainer")

In [ ]:
# Fallback to the model variable
model.save_pretrained('/kaggle/working')
print("✅ Adapter saved from model")

In [ ]:
from IPython.display import FileLink, display

# Create download links
display(FileLink('adapter_model.safetensors'))
display(FileLink('lora_adapter.zip'))

## 9. Phase 2 — GRPO Reinforcement Learning

### 9a. Reward Functions

Three reward signals:
| Signal | Weight | Rationale |
|---|---|---|
| `reward_correct_answer` | +1.0 | Primary: exact match or numerical tolerance |
| `reward_has_boxed` | +0.2 | Format compliance |
| `reward_length` | ±0.05–0.1 | Penalise trivially short or verbose responses |

In [ ]:
# TRL 0.29 reward signature: (prompts, completions, **kwargs)
# Extra dataset columns are passed through **kwargs

def reward_correct_answer(prompts, completions, **kwargs):
    """Primary reward: +1.0 if \\boxed{} answer matches ground truth."""    answers = kwargs.get('answer', [None] * len(completions))
    rewards = []
    for comp, ans in zip(completions, answers):
        pred = extract_boxed(comp)
        if pred is None or ans is None:
            rewards.append(0.0); continue
        if pred.strip() == str(ans).strip():
            rewards.append(1.0); continue
        try:
            p_f, a_f = float(pred), float(ans)
            # Relative tolerance 1e-4 for floating-point answers
            rewards.append(1.0 if abs(p_f - a_f) / max(abs(a_f), 1e-9) < 1e-4 else 0.0)
        except Exception:
            rewards.append(0.0)
    return rewards


def reward_has_boxed(prompts, completions, **kwargs):
    """Format reward: +0.2 if \\boxed{} present in completion."""    return [0.2 if re.search(r'\\boxed\{[^}]+\}', c) else 0.0 for c in completions]


def reward_length(prompts, completions, **kwargs):
    """Length shaping: penalise trivially short or extremely long responses."""    rewards = []
    for c in completions:
        n = len(c.split())
        if n < 20:
            rewards.append(-0.1)   # too short → likely no reasoning
        elif n > 1500:
            rewards.append(-0.05)  # too long → verbosity penalty
        else:
            rewards.append(0.0)
    return rewards


print('✅ Reward functions ready')


### 9b. GRPO Dataset

In [ ]:
def make_grpo_row(row: dict) -> dict:
    """Format a training row as a chat prompt (no assistant turn — model completes it)."""    user_msg = row['prompt'] + '\nPut your final answer inside \\boxed{}.'
    messages = [{'role': 'user', 'content': user_msg}]
    try:
        prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        prompt_text = f'<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n'
    return {'prompt': prompt_text, 'answer': str(row['answer'])}


grpo_dataset = Dataset.from_list(
    [make_grpo_row(row) for row in train_df[['prompt', 'answer']].to_dict('records')]
)
print(f'GRPO dataset: {len(grpo_dataset)} samples')
print('Sample prompt (first 300 chars):')
print(grpo_dataset[0]['prompt'][:300])


### 9c. GRPO Training

In [ ]:
# Fix 5 must remain active for GRPO as well
os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = '/tmp/ptxas-blackwell'
nv_compiler.get_ptxas_version = lambda arch: '12.0'

grpo_config = GRPOConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = GRPO_EPOCHS,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    num_generations             = GRPO_NUM_GEN,   # completions per prompt
    max_completion_length       = GRPO_MAX_COMP_LEN,
    max_prompt_length           = 512,
    beta                        = 0.001,           # KL penalty coefficient
    learning_rate               = GRPO_LR,
    lr_scheduler_type           = 'cosine',
    warmup_ratio                = 0.05,
    bf16                        = True,
    logging_steps               = 5,
    save_strategy               = 'no',
    optim                       = 'adamw_torch',
    max_grad_norm               = 0.1,
    report_to                   = 'none',
    use_vllm                    = False,           # vLLM not available on this image
)

grpo_trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    args             = grpo_config,
    train_dataset    = grpo_dataset,
    reward_funcs     = [reward_correct_answer, reward_has_boxed, reward_length],
)

print('🎯 Starting GRPO...')
grpo_trainer.train()
print('✅ GRPO complete')


## 10. Save Adapter & Package submission.zip

In [ ]:
# ─── Save the LoRA adapter weights ───────────────────────────────────────────
print(f'Saving adapter to {OUTPUT_DIR}...')
sft_trainer.model.save_pretrained(OUTPUT_DIR)

# ─── Remove checkpoint subdirs to save disk space ────────────────────────────
for item in os.listdir(OUTPUT_DIR):
    item_path = os.path.join(OUTPUT_DIR, item)
    if os.path.isdir(item_path) and item.startswith('checkpoint-'):
        shutil.rmtree(item_path)
        print(f'Removed checkpoint: {item}')

# ─── List saved files ─────────────────────────────────────────────────────────
print(f'\nFiles in {OUTPUT_DIR}:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fpath):
        print(f'  {f:55s}  {os.path.getsize(fpath)/1024/1024:.1f} MB')


In [ ]:
# ─── Validate adapter_config.json ────────────────────────────────────────────
config_path = os.path.join(OUTPUT_DIR, 'adapter_config.json')
assert os.path.exists(config_path), '❌ adapter_config.json missing'

with open(config_path) as f:
    cfg = json.load(f)

rank = cfg.get('r', 999)
assert rank <= 32, f'❌ LoRA rank {rank} exceeds competition limit of 32'
print(json.dumps(cfg, indent=2))
print(f'\n✅ LoRA rank = {rank} (≤ 32 ✓)')


In [ ]:
# ─── Build submission.zip ─────────────────────────────────────────────────────
zip_path     = '/kaggle/working/submission.zip'
files_to_zip = ['adapter_model.safetensors', 'adapter_config.json', 'README.md']

print(f'Creating {zip_path}...')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in files_to_zip:
        fpath = os.path.join(OUTPUT_DIR, fname)
        if os.path.exists(fpath):
            zf.write(fpath, fname)
            print(f'  Added: {fname}')
        else:
            print(f'  ⚠️  {fname} not found — skipped')

zip_size = os.path.getsize(zip_path) / 1024 / 1024
print(f'\nArchive size: {zip_size:.1f} MB')

# ─── Validate the zip ─────────────────────────────────────────────────────────
with zipfile.ZipFile(zip_path, 'r') as zf:
    names = zf.namelist()
    print(f'Contents: {names}')
    assert 'adapter_config.json'       in names, '❌ adapter_config.json missing from zip'
    assert 'adapter_model.safetensors' in names, '❌ adapter_model.safetensors missing from zip'

print('\n✅ submission.zip is valid and ready for submission')
print(f'   Path: {zip_path}')


---
## ✅ Final Checklist

| Check | Detail |
|---|---|
| Only `datasets` and `trl` installed from wheels | All other packages from pre-installed Docker image |
| `--ignore-installed` flag on pip | Forces correct version despite pre-existing installs |
| **Fix 1:** RMSNorm bypassed | Pure-PyTorch fallback, no Triton kernel compilation |
| **Fix 2:** PTXAS copied to /tmp | Requires `ryanholbrook/nvidia-utility-script` data source |
| **Fix 3:** Mamba3 stubbed | Prevents cutlass C++ crash on SM 12.0 |
| **Fix 4:** `is_fast_path_available = False` | Applied after model load, before LoRA attach |
| **Fix 5:** Triton env vars + version spoof | Set before **each** `trainer.train()` call (SFT and GRPO) |
| `kagglehub.model_download()` | Accesses Kaggle model registry — works offline |
| No BitsAndBytes quantisation | Full bfloat16 — 96 GB VRAM has no memory pressure |
| LoRA target regex `in_proj\|out_proj\|up_proj\|down_proj` | Nemotron-H SSM hybrid layers, not standard transformer |
| `processing_class=tokenizer` | TRL 0.29 correct API (`tokenizer=` deprecated) |
| `optim='adamw_torch'` | No BNB dependency |
| `gradient_checkpointing_kwargs={'use_reentrant': False}` | Stable gradient checkpointing |
| GRPO `beta=0.001` | Conservative KL penalty — prevents policy collapse |
| GRPO `num_generations=8` | 8 completions sampled per prompt for relative ranking |
| Manual zip with exactly 3 files | `adapter_model.safetensors`, `adapter_config.json`, `README.md` |
| LoRA rank ≤ 32 verified | Assertion before zip creation |
